# FastAPI 模型服务教程

本教程详细介绍如何使用 FastAPI 构建模型推理服务，包括：

1. **FastAPI 基础**: 创建 REST API 服务
2. **请求/响应模型**: Pydantic 数据验证
3. **异步推理**: 高并发处理
4. **动态批处理**: 提高吞吐量
5. **健康检查和监控**: 生产级服务

---

## 为什么选择 FastAPI？

```
FastAPI 优势:
┌─────────────────────────────────────────┐
│  高性能: 基于 Starlette 和 Pydantic     │
│  自动文档: Swagger UI 和 ReDoc          │
│  类型安全: 完整的类型提示支持            │
│  异步支持: 原生 async/await             │
│  易于使用: 简洁的 API 设计              │
└─────────────────────────────────────────┘
```

In [ ]:
import sys
import os
sys.path.insert(0, '../src')

import numpy as np
import time
import asyncio

# 检查 FastAPI
try:
    from fastapi import FastAPI
    from pydantic import BaseModel
    print("FastAPI 已安装")
    FASTAPI_AVAILABLE = True
except ImportError:
    print("FastAPI 未安装")
    print("安装命令: pip install fastapi uvicorn")
    FASTAPI_AVAILABLE = False

# 导入封装模块
from fastapi_server import (
    FASTAPI_AVAILABLE as MODULE_FASTAPI_AVAILABLE,
    MetricsCollector,
    InferenceCache,
    DynamicBatcher,
)

print(f"\n模块 FastAPI 可用: {MODULE_FASTAPI_AVAILABLE}")

## 1. FastAPI 基础

### 1.1 创建简单的 API

In [ ]:
if FASTAPI_AVAILABLE:
    # 创建 FastAPI 应用
    app = FastAPI(
        title="Model Inference API",
        description="模型推理服务 API",
        version="1.0.0"
    )
    
    # 定义路由
    @app.get("/")
    async def root():
        return {"message": "Welcome to Model Inference API"}
    
    @app.get("/health")
    async def health():
        return {"status": "healthy"}
    
    print("FastAPI 应用创建成功!")
    print(f"路由: {[route.path for route in app.routes]}")
else:
    print("跳过 FastAPI 示例")

### 1.2 请求/响应模型

使用 Pydantic 定义数据模型，实现自动验证和文档生成。

In [ ]:
if FASTAPI_AVAILABLE:
    from pydantic import BaseModel, Field
    from typing import List, Optional
    
    # 请求模型
    class PredictRequest(BaseModel):
        """预测请求"""
        data: List[float] = Field(..., description="输入数据")
        batch_size: Optional[int] = Field(1, description="批次大小")
        
        class Config:
            json_schema_extra = {
                "example": {
                    "data": [1.0, 2.0, 3.0, 4.0],
                    "batch_size": 1
                }
            }
    
    # 响应模型
    class PredictResponse(BaseModel):
        """预测响应"""
        prediction: List[float] = Field(..., description="预测结果")
        confidence: Optional[float] = Field(None, description="置信度")
        latency_ms: float = Field(..., description="推理延迟(毫秒)")
    
    # 测试模型
    request = PredictRequest(data=[1.0, 2.0, 3.0, 4.0])
    print(f"请求数据: {request.data}")
    print(f"批次大小: {request.batch_size}")
    
    response = PredictResponse(
        prediction=[0.8, 0.15, 0.05],
        confidence=0.8,
        latency_ms=5.2
    )
    print(f"\n响应: {response.model_dump()}")
else:
    print("跳过 Pydantic 示例")

## 2. 指标收集器

MetricsCollector 用于收集服务指标，包括请求计数、延迟统计等。

In [ ]:
# 创建指标收集器
metrics = MetricsCollector(max_history=1000)

print("MetricsCollector 功能:")
print("  - 记录请求延迟")
print("  - 统计成功/失败请求")
print("  - 计算 p50/p99 延迟")

In [ ]:
# 模拟记录请求
async def simulate_requests():
    """模拟请求并记录指标"""
    for i in range(100):
        # 模拟不同的延迟
        latency = np.random.exponential(10)  # 指数分布
        success = np.random.random() > 0.05  # 95% 成功率
        await metrics.record_request(latency, success)

# 运行模拟
await simulate_requests()

# 获取指标
stats = metrics.get_metrics()
print("服务指标:")
print(f"  总请求数: {stats['total_requests']}")
print(f"  成功请求: {stats['successful_requests']}")
print(f"  失败请求: {stats['failed_requests']}")
print(f"  平均延迟: {stats['avg_latency_ms']:.2f} ms")
print(f"  P50 延迟: {stats['p50_latency_ms']:.2f} ms")
print(f"  P99 延迟: {stats['p99_latency_ms']:.2f} ms")

## 3. 推理缓存

InferenceCache 实现 LRU 缓存，避免重复计算相同输入。

In [ ]:
# 创建缓存
cache = InferenceCache(max_size=100, ttl=60.0)  # 最大 100 条，60 秒过期

print("InferenceCache 功能:")
print("  - LRU 淘汰策略")
print("  - TTL 过期机制")
print("  - 输入哈希去重")

In [ ]:
# 模拟推理函数
def slow_inference(data):
    """模拟耗时推理"""
    time.sleep(0.1)  # 模拟 100ms 推理
    return [sum(data) / len(data)]  # 返回平均值

# 带缓存的推理
def cached_inference(data):
    """带缓存的推理"""
    # 检查缓存
    cached = cache.get(data)
    if cached is not None:
        return cached, True  # 缓存命中
    
    # 执行推理
    result = slow_inference(data)
    
    # 存入缓存
    cache.set(data, result)
    return result, False  # 缓存未命中

# 测试缓存效果
test_data = [1.0, 2.0, 3.0, 4.0, 5.0]

# 第一次调用 (缓存未命中)
start = time.time()
result1, hit1 = cached_inference(test_data)
time1 = (time.time() - start) * 1000
print(f"第一次调用: {time1:.2f}ms, 缓存命中: {hit1}, 结果: {result1}")

# 第二次调用 (缓存命中)
start = time.time()
result2, hit2 = cached_inference(test_data)
time2 = (time.time() - start) * 1000
print(f"第二次调用: {time2:.2f}ms, 缓存命中: {hit2}, 结果: {result2}")

print(f"\n加速比: {time1/time2:.1f}x")

## 4. 动态批处理

DynamicBatcher 将多个请求合并为批次，提高 GPU 利用率。

```
单请求处理:                    批处理:
┌─────┐                       ┌─────┐
│Req 1│→ Infer → Response     │Req 1│─┐
└─────┘                       └─────┘ │
┌─────┐                       ┌─────┐ │    ┌──────────┐
│Req 2│→ Infer → Response     │Req 2│─┼──→ │  Batch   │→ Responses
└─────┘                       └─────┘ │    │  Infer   │
┌─────┐                       ┌─────┐ │    └──────────┘
│Req 3│→ Infer → Response     │Req 3│─┘
└─────┘                       └─────┘

3 次推理                       1 次推理
```

In [ ]:
# 定义批量预测函数
def batch_predict(batch_data):
    """批量预测函数"""
    # 模拟批量推理
    time.sleep(0.05)  # 50ms 批量推理
    return [sum(data) for data in batch_data]

# 创建动态批处理器
batcher = DynamicBatcher(
    predict_fn=batch_predict,
    max_batch_size=8,
    max_wait_time=0.02  # 20ms 最大等待
)

print("DynamicBatcher 配置:")
print(f"  最大批次大小: {batcher.max_batch_size}")
print(f"  最大等待时间: {batcher.max_wait_time * 1000}ms")

In [ ]:
# 测试动态批处理
async def test_dynamic_batching():
    """测试动态批处理"""
    # 并发发送多个请求
    tasks = [
        batcher.add_request([1.0, 2.0]),
        batcher.add_request([3.0, 4.0]),
        batcher.add_request([5.0, 6.0]),
        batcher.add_request([7.0, 8.0]),
    ]
    
    start = time.time()
    results = await asyncio.gather(*tasks)
    elapsed = (time.time() - start) * 1000
    
    print(f"批处理结果: {results}")
    print(f"总耗时: {elapsed:.2f}ms")
    print(f"平均每请求: {elapsed/len(tasks):.2f}ms")

await test_dynamic_batching()

## 5. 完整的模型服务器

使用封装的 ModelServer 类创建完整的推理服务。

In [ ]:
if MODULE_FASTAPI_AVAILABLE:
    from fastapi_server import ModelServer, create_model_server
    
    # 定义预测函数
    def my_predict(data):
        """自定义预测函数"""
        # 模拟模型推理
        result = np.array(data) * 2
        return result.tolist()
    
    # 创建服务器
    server = create_model_server(
        predict_fn=my_predict,
        name="demo-server",
        enable_batching=True,
        max_batch_size=16,
        enable_cache=True,
        cache_size=500
    )
    
    print("ModelServer 创建成功!")
    print(f"  服务名称: {server.name}")
    print(f"  批处理: {server.enable_batching}")
    print(f"  缓存: {server.enable_cache}")
    print(f"\n可用端点:")
    for route in server.app.routes:
        if hasattr(route, 'path'):
            print(f"  {route.path}")
else:
    print("跳过 ModelServer 示例 (FastAPI 未安装)")

## 6. 启动服务 (示例代码)

以下代码展示如何启动服务器（在 Notebook 中不执行）：

In [ ]:
print("启动服务器示例代码:")
print("""
# main.py
from fastapi_server import create_model_server
import numpy as np

def predict(data):
    # 你的模型推理逻辑
    return np.array(data).mean()

server = create_model_server(
    predict_fn=predict,
    name="my-model-server",
    enable_batching=True,
    enable_cache=True
)

if __name__ == "__main__":
    server.run(host="0.0.0.0", port=8000)

# 启动命令:
# python main.py
# 或
# uvicorn main:server.app --host 0.0.0.0 --port 8000 --workers 4
""")

## 7. 客户端调用示例

In [ ]:
print("客户端调用示例:")
print("""
import httpx

# 同步调用
with httpx.Client() as client:
    response = client.post(
        "http://localhost:8000/predict",
        json={"data": [1.0, 2.0, 3.0, 4.0]}
    )
    result = response.json()
    print(f"预测结果: {result['prediction']}")
    print(f"延迟: {result['latency_ms']}ms")

# 异步调用
async with httpx.AsyncClient() as client:
    response = await client.post(
        "http://localhost:8000/predict",
        json={"data": [1.0, 2.0, 3.0, 4.0]}
    )
    result = response.json()

# 健康检查
response = httpx.get("http://localhost:8000/health")
print(response.json())  # {"status": "healthy", ...}

# 获取指标
response = httpx.get("http://localhost:8000/metrics")
print(response.json())  # {"total_requests": 100, ...}
""")

## 总结

本教程介绍了 FastAPI 模型服务的核心功能：

1. **FastAPI 基础**: 创建 REST API 和路由
2. **Pydantic 模型**: 请求/响应数据验证
3. **指标收集**: 监控服务性能
4. **推理缓存**: 避免重复计算
5. **动态批处理**: 提高吞吐量

### 最佳实践

- 使用 Pydantic 模型进行数据验证
- 启用缓存减少重复计算
- 使用动态批处理提高 GPU 利用率
- 实现健康检查支持 Kubernetes 部署
- 收集指标用于监控和告警